# Analiza wyników — LLM Security Sandbox

Notebook ładuje `data/results/results.csv` (wygenerowany przez `python -m scripts.run_attacks`) i tworzy:
1. tabelę pivot **model × atak** (1 = atak skuteczny),
2. ranking **najgroźniejszych kategorii** ataków,
3. porównanie **bez** vs **z** warstwą obronną (jeśli istnieje `results_shielded.csv`),
4. wykresy słupkowe.

> Kontekst prawno-etyczny: wykresy i tabele to materiał dowodowy do oceny należytej staranności Deployera (EU AI Act art. 12, 15; AILD art. 3–4).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'data').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'data' / 'results' / 'results.csv'
SHIELDED = ROOT / 'data' / 'results' / 'results_shielded.csv'
print('ROOT =', ROOT)
print('results.csv exists:', RESULTS.exists())
print('results_shielded.csv exists:', SHIELDED.exists())

In [ ]:
df = pd.read_csv(RESULTS)
print('Wierszy:', len(df), '| Modele:', df['Model'].unique().tolist())
df.head()

## 1. Pivot model × atak (1 = atak skuteczny)

In [ ]:
pivot = df.pivot_table(index='Model', columns='Attack_ID', values='Attack_Success', aggfunc='max').fillna(0).astype(int)
pivot['SUMA'] = pivot.sum(axis=1)
pivot

## 2. Skuteczność ataków per model

In [ ]:
per_model = df.groupby('Model')['Attack_Success'].mean().mul(100).round(1).sort_values(ascending=False)
ax = per_model.plot.bar(figsize=(7, 4), edgecolor='black')
ax.set_ylabel('% skutecznych ataków')
ax.set_title('Podatność modelu na 15 ataków Prompt Injection')
for i, v in enumerate(per_model.values):
    ax.text(i, v + 1, f'{v}%', ha='center')
plt.tight_layout(); plt.show()
per_model

## 3. Najgroźniejsze kategorie ataków

In [ ]:
per_cat = df.groupby('Category')['Attack_Success'].mean().mul(100).round(1).sort_values(ascending=False)
ax = per_cat.plot.barh(figsize=(7, 5), edgecolor='black', color='tomato')
ax.set_xlabel('% skuteczności (średnia po modelach)')
ax.set_title('Skuteczność ataków wg kategorii')
plt.tight_layout(); plt.show()
per_cat

## 4. Cele ataków: jailbreak vs secret_exfiltration

In [ ]:
df.groupby(['Model', 'Goal'])['Attack_Success'].mean().mul(100).round(1).unstack().fillna(0)

## 5. Porównanie z warstwą obronną LLM-Guard
Wymaga uruchomienia: `python -m scripts.run_with_shield`.

In [ ]:
if SHIELDED.exists():
    sh = pd.read_csv(SHIELDED)
    blocked_pct = (sh['Shield_Status'] == 'BLOCKED').mean() * 100
    print(f'LLM-Guard zablokował: {blocked_pct:.1f}% ataków')
    display(sh[['Attack_ID','Category','Shield_Status','Risk_Score','Shield_Reason']])
else:
    print('Brak results_shielded.csv — uruchom scripts/run_with_shield.py')

## 6. Wnioski (uzupełnij w `wyniki/raport.md`)
- Który model jest najsłabszy / najmocniejszy?
- Które kategorie ataków przebijają model bez warstwy obronnej?
- O ile spada skuteczność po włączeniu LLM-Guard?
- Czy zaobserwowane wycieki sekretu byłyby naruszeniem RODO art. 5 ust. 1 lit. f?